# POC pengiring musik di JupyterLab lokal

Mulai dari model pretrained, bukan training dari nol. Kita menilai bass/drum dari
rekaman gitar/piano, lalu memutuskan apakah fine-tuning diperlukan.

**Jalankan dari atas.** Default notebook hanya memeriksa hardware dan menjalankan
persiapan/plan. Instalasi ML, unduhan bobot dan inferensi memiliki sakelar terpisah
agar Run All pertama tidak tiba-tiba mengunduh puluhan GB.
Tidak ada gerbang review lisensi untuk POC ini. Tidak ada koneksi ke filesystem
lokal dari agen; GitHub menyinkronkan kode, sementara audio dan bobot tetap lokal.


In [ ]:
from pathlib import Path
import json, subprocess, sys, uuid
from datetime import datetime, timezone

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "tools/stem_eval").is_dir(), "Buka notebook dari checkout StemsGenerating-Models"
sys.path.insert(0, str(ROOT / "scripts"))
sys.path.insert(0, str(ROOT / "tools/stem_eval"))
from stream_process import run_logged
from diagnose import diagnose
from setup_runtime import runtime_python, UPSTREAM

REPORTS = ROOT / "reports"
REPORTS.mkdir(exist_ok=True)
HARDWARE = diagnose(ROOT)
(REPORTS / "hardware.json").write_text(json.dumps(HARDWARE, indent=2))
print(json.dumps(HARDWARE, indent=2))


## 1. Pilih konfigurasi

Jika belum tahu GPU, gunakan `MODEL = "auto"`. Rekomendasi XL memakai VRAM yang
terdeteksi kelas 24 GB (≥24000 MiB yang dilaporkan driver), mengikuti panduan upstream; ini bukan jaminan bebas OOM.
XL adalah kandidat kualitas lebih tinggi menurut upstream, belum pemenang pada
klipmu. CPU offload mengurangi kebutuhan VRAM dengan kemungkinan waktu lebih lama.

`SOURCE = None` membuat fixture sinus untuk mengecek pipeline saja. Untuk evaluasi
musikal, isi path rekaman gitar/piano asli. Jika memakai ekspor SongMap, isi
`CONDITIONING` dan gunakan `START = 0` karena WAV sudah dipotong.

Pada Apple Silicon mulai dengan Base dan satu seed (`SEEDS = [0]`) sebelum mencoba
XL. RAM terpadu ditampilkan di diagnostik; ruang disk saja tidak menentukan apakah
model muat. Ketersediaan MPS baru dipastikan setelah runtime model terpasang.


In [ ]:
MODEL = "auto"  # atau "acestep-v15-xl-base" / "acestep-v15-base"
INSTRUMENT = "bass"  # "bass" atau "drums"
SEEDS = [0] if HARDWARE.get("apple_silicon") else [0, 1, 2]
OFFLOAD = True
DEVICE = "auto"  # MPS pada Apple Silicon, CUDA pada NVIDIA; diperiksa di runtime model
SOURCE = None  # contoh: ROOT / "data" / "guitar.wav"
CONDITIONING = None  # contoh: ROOT / "data" / "conditioning.json"
START = 0.0
DURATION = 10.0
INSTALL_RUNTIME = False
DOWNLOAD_WEIGHTS = False
GENERATE = False

SELECTED_MODEL = HARDWARE["suggested_model"] if MODEL == "auto" else MODEL
assert SELECTED_MODEL in {"acestep-v15-base", "acestep-v15-xl-base"}
PROFILE = "xl-base" if SELECTED_MODEL.endswith("xl-base") else "base"
ASSETS = ROOT / "configs" / f"assets-{PROFILE}.json"
manifest = json.loads(ASSETS.read_text())
print("Model:", SELECTED_MODEL, "| download:", round(manifest["download_bytes"] / 1e9, 2), "GB")
print("Cadangkan ruang kosong minimal", round((2*manifest["download_bytes"] + 5*1024**3)/1024**3, 1), "GiB untuk unduhan penuh")
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:6]
WORK = ROOT / "runs" / RUN_ID
WORK.mkdir(parents=True)


## 2. Persiapkan satu klip tanpa model

Perlu `ffmpeg` dan `ffprobe`. Di Terminal Mac jalankan `brew install ffmpeg`,
lalu restart kernel. Helper juga mencari `/opt/homebrew/bin` dan `/usr/local/bin`
jika PATH Jupyter belum diperbarui. Tidak ada tempo/chord palsu yang dibuat untuk fixture.


In [ ]:
import array, math, wave
from local_clip import local_clip
from system_tools import require_audio_tools
require_audio_tools()

if SOURCE is None:
    fixture = WORK / "fixture.wav"
    values = array.array("h", (round(3000*math.sin(2*math.pi*220*i/44100)) for i in range(441000)))
    if sys.byteorder != "little":
        values.byteswap()
    with wave.open(str(fixture), "wb") as stream:
        stream.setparams((1, 2, 44100, 0, "NONE", "not compressed"))
        stream.writeframes(values.tobytes())
    source = fixture
    print("FIXTURE SINTETIS: hanya validasi pipeline, bukan penilaian musikal")
else:
    source = Path(SOURCE).expanduser().resolve()
    assert source.is_file(), f"Audio tidak ditemukan: {source}"
PREPARED = WORK / "prepared"
local_clip(source, PREPARED, START, DURATION, Path(CONDITIONING) if CONDITIONING else None)
print((PREPARED / "prepared.json").read_text())


In [ ]:
from IPython.display import Audio, display
display(Audio(filename=str(PREPARED / "source.wav")))


## 3. Buat rencana seed yang sama

Status `planned` tidak berarti inferensi sudah dijalankan. ACE memakai sumber
bersih; JSON beat/chord disimpan untuk evaluasi, tidak dianggap sebagai input
model yang sudah didukung.


In [ ]:
PLAN = WORK / "plan"
base_command = [sys.executable, str(ROOT / "tools/stem_eval/run.py"),
    "--prepared", str(PREPARED), "--backend", "ace", "--instrument", INSTRUMENT,
    "--seeds", *map(str, SEEDS), "--ace-model", SELECTED_MODEL, "--device", DEVICE]
run_logged([*base_command, "--out", str(PLAN)], check=True)
print((PLAN / "experiment.json").read_text())


## 4. Instal runtime terpisah (sekali)

Ubah `INSTALL_RUNTIME = True` pada sel konfigurasi, jalankan ulang konfigurasi dan
sel persiapan sebelum melanjutkan. Script mengambil source pada commit tetap dan
menjalankan `uv sync --frozen --python 3.11`. Kernel Jupyter tetap memakai Python
semula; inferensi dipanggil lewat Python `.venv` upstream.

Instalasi ini besar dan belum dijalankan pada hardware lokalmu. Jika gagal,
simpan pesan kesalahan lengkap. Jangan ubah lockfile atau matikan pemeriksaan TLS.

Python 3.15 JupyterLab tidak perlu diganti atau dipasangi PyTorch. `uv` menyiapkan
Python 3.11 pada `vendor/ACE-Step-1.5/.venv` dan melaporkan CUDA serta MPS di situ.


In [ ]:
if INSTALL_RUNTIME:
    run_logged([sys.executable, str(ROOT / "scripts/setup_runtime.py"), "--install"], check=True)
    run_logged([str(runtime_python()), str(ROOT / "scripts/diagnose.py")], cwd=ROOT, check=True)
else:
    print("Instalasi dilewati. Aktifkan INSTALL_RUNTIME jika hardware dan ruang disk sudah sesuai.")


## 5. Unduh dan verifikasi model lokal

Ubah `DOWNLOAD_WEIGHTS = True` setelah runtime siap. Manifest berisi revisi HF
dan SHA-256 LFS / hash Git untuk setiap file. Total bobot saat disiapkan sekitar
14.88 GB (Base) atau 30.05 GB (XL + komponen utama). Nilai tepat tampil di atas.
Komponen Turbo/LM ikut disediakan karena pemeriksaan initializer upstream saat
ini memerlukannya, walaupun baseline memakai Base tanpa planner.

Ini mengunduh kode Python model juga. Kode berasal dari revisi resmi yang dipin.
File yang tidak cocok checksum ditolak, bukan diterima dengan checksum baru.

Output sekarang menampilkan nomor file, ukuran, progress transfer HF bila tersedia,
verifikasi checksum dan selesai/gagal. Jika child tidak memberi output selama
15 detik, pesan status menunjukkan proses masih hidup; pesan itu bukan bukti
byte jaringan terus bertambah. [*] berarti sel belum selesai.


In [ ]:
if DOWNLOAD_WEIGHTS:
    assert runtime_python().is_file(), "Jalankan instalasi runtime dahulu"
    run_logged([str(runtime_python()), str(ROOT / "scripts/model_assets.py"),
                    "--manifest", str(ASSETS), "--upstream", str(UPSTREAM), "--download"], check=True)
else:
    print("Unduhan dilewati. Tidak ada bobot yang diunduh oleh Run All default.")


## 6. Generate bass/drum dan ukur

Ubah `GENERATE = True` setelah runtime dan aset siap. Gunakan rekaman asli untuk
membandingkan kualitas. Runner memilih MPS pada Mac Apple Silicon atau CUDA pada NVIDIA.
Python model tetap 3.11, walaupun kernel Jupyter memakai 3.15. Tidak ada fallback
CPU diam-diam. MPS adalah runtime Mac, bukan bukti dukungan Core AI/iPhone.

Baseline: Base/XL Base `lego`, 50 langkah, CFG 7, planner/CoT mati, satu stem per
seed. `OFFLOAD` tidak mengubah pilihan model. Uji satu klip dahulu sebelum batch.
Model yang lebih besar belum tentu lebih baik mengikuti groove; dengarkan hasil.


In [ ]:
RESULTS = WORK / "generated"
if GENERATE:
    assert SOURCE is not None, "Pilih rekaman gitar/piano asli sebelum menilai generasi"
    assert runtime_python().is_file(), "Runtime belum dipasang"
    command = [str(runtime_python()), *base_command[1:], "--out", str(RESULTS),
               "--execute", "--upstream", str(UPSTREAM), "--assets-manifest", str(ASSETS)]
    if OFFLOAD:
        command.append("--offload")
    run_logged(command, check=True)
else:
    print("Inferensi dilewati. Plan dan fixture bukan hasil model.")


## 7. Dengarkan dan kirim laporan kecil lewat bridge

Mix selalu memakai sumber bersih; click hanya konteks STAGE pada eksperimen
terpisah. Volume mix 0.5 × sumber + 0.5 × stem. Nilai harmoni/groove, dinamika,
naturalness, dan kebocoran sumber. Waktu MPS/CUDA bukan benchmark Core AI/iPhone. Metrik MPS yang dicatat adalah
current allocator/driver memory, bukan peak atau total memori proses.

Audio, bobot dan folder `runs/` tidak masuk Git. Salin hanya hardware report,
summary.csv, result.json (setelah ditinjau) ke `feedback/` dan commit/push bila ingin
membagikannya. Notebook menyimpan output kosong di repository; hapus output sebelum
commit agar audio tidak tertanam ke notebook. Jangan kirim token atau isi `.env`.


In [ ]:
from summarize import summarize
if RESULTS.exists() and list(RESULTS.glob("*/result.json")):
    summarize(RESULTS)
    for audio in sorted(RESULTS.glob("*/mix.wav")):
        print(audio.parent.name)
        display(Audio(filename=str(audio)))
else:
    print("Belum ada hasil inferensi nyata. Hardware report:", REPORTS / "hardware.json")
